<a href="https://colab.research.google.com/github/ZainDev04/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZainDev04/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb)

**Lane 3 — Structured Content Archetype Clustering** (provisional; carried from ML-02 and ML-03).

ML-02 and ML-03 built the archetype framing on the 30k starter CSV and closed with a specific complaint:
a trailing-90-day aggregate cannot tell a page that is *becoming* an archetype from one that has been it
for a year. This notebook moves the contract onto the **warehouse daily table**, where the observation
window is mine to choose day by day.

> **Before Runtime → Run all:** request access on
> [`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)
> (instant), create a **plain Read** token in HF settings, and store it in Colab's 🔑 **Secrets** panel as
> `HF_TOKEN`. Never paste a token into a cell — this repo is public.
>
> Skills for this card: `skills/writing-data-contracts/SKILL.md` + `skills/flyrank/flyrank-data/SKILL.md`.
> Expect **~3–6 minutes** on the aggregation cell; everything else is seconds.

**Month discipline.** `fact_content_daily_performance_sample` is *not* a random sample — it is exactly the
final month (June 2026). I develop on **`month=2026-03`** and leave June sealed, so that when I later want
to ask whether these archetypes are stable over time I still have a month I have never looked at.

In [1]:
%pip -q install duckdb huggingface_hub

In [2]:
import os, getpass

# Colab Secrets first, then environment, then a masked prompt. The token is never printed.
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN") or getpass.getpass("HF READ token (hf_...): ")
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "No usable token found - set HF_TOKEN in Colab Secrets."
print("Token loaded (not shown).")

Token loaded (not shown).


In [3]:
import duckdb, pandas as pd, numpy as np

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"                       # my observation window - mid-panel, well clear of June

FACT = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

# Setup smoke-test (NOT one of the three contract queries): what columns actually exist?
schema = con.sql(f"DESCRIBE SELECT * FROM {FACT} LIMIT 1").df()
print(f"fact_content_daily_performance, month={MONTH} - {len(schema)} columns\n")
print(", ".join(schema["column_name"].tolist()))

fact_content_daily_performance, month=2026-03 - 31 columns

report_date, client_hash_id, content_hash_id, client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available, gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position, ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions, ga4_total_engagement_sec, sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other, scroll_events, month


## 1. Unit of analysis + time window

### The contract, in five plain answers

**1. What one row means for my lane.**
In the *source* table, one row = **one content item, for one client, on one day** — grain is
`report_date + client_hash_id + content_hash_id`. In my *archetype frame*, one row =
**one content item (a page), summarised across the whole observation month**. The page is the unit because
the page is what receives a treatment. A cluster is a property I attach to a page, never a row in its own
right.

**2. Which tables I use.**
`fact_content_daily_performance`, one month partition, and nothing else for the features.
`dim_clients` is read for context only — history coverage — and never joined into the clustering frame.

I skip `fact_content_query_90d` deliberately. Its window is a fixed trailing 90 days that does not line up
with my one-month observation window, so joining it would describe a page's query mix over a *different
period* than the behaviour I am clustering. For a supervised lane that misalignment is leakage; for a
descriptive lane it is worse in a quieter way — it produces a profile that is not a profile of anything.

**3. Which time window.**

```text
OBSERVATION WINDOW
month = 2026-03
2026-03-01 .. 2026-03-31
```

One window, not two. This is the structural difference between an unsupervised lane and every supervised
one in this internship: **there is no target window, so there is nothing for a feature window to leak
into.** What replaces that discipline is a different rule, and it is just as strict —

> **Every column must be measured inside the window the archetype claims to describe.**

If I mix a March behaviour with an all-time content property, the resulting group is not "how this page
behaved in March"; it is a blend of two periods wearing one name. Section 3's feature table enforces that.

**4. What I would predict or rank (label or proxy).**
**Neither. There is no label and nothing is predicted.** The output is a cluster assignment per page plus
an action mapped to each cluster.

The closest thing to a proxy is the **action mapping** — protect / improve / rewrite / merge / prune /
monitor — because that is the part that can be wrong in a way someone would notice. ML-03 committed to a
falsifiable version of it: *if two clusters earn the same action, they did not need to be separate
clusters.* That is what stands in for accuracy here.

**5. One thing I deliberately exclude.**
**Any pre-computed grouping of the pages — including my own hand-written archetype ladder from ML-07.**
Clustering on a grouping I already wrote down rediscovers my own rule and reports it as a discovery. It is
the unsupervised form of the circular-result trap, it produces a beautiful score, and section 3 does it on
purpose so I can watch it happen.

*(Runner-up exclusion: `client_hash_id`. It is the thing I use to CHECK whether a cluster is really one
client's house style. The moment it becomes a feature, that check is meaningless.)*

## 2. Fields: feature / label / context / excluded

| Column | Bucket | Why |
|---|---|---|
| `report_date` | context | defines the observation window; never fed to the model |
| `client_hash_id` | context | pseudonym; used to test whether clusters are house styles. **Never a feature** |
| `content_hash_id` | context | pseudonym; row key and join key only |
| `gsc_impressions` (March) | **feature** | exposure, measured inside the window |
| `gsc_clicks` (March) | **feature** | demand captured, measured inside the window |
| CTR (March clicks ÷ March impressions) | **feature** | derived from two in-window columns only |
| `gsc_avg_position` (March, impression-weighted) | **feature** | where it ranked *during the window*, not an all-time mean |
| days with impressions ÷ days in month | **feature** | steadiness of exposure; in-window by construction |
| `ga4_data_available` | context | availability flag; decides which rows are trustworthy (Query 3) |
| GA4 columns (sessions, engagement, scroll) | **excluded** | zero-*filled*, not zero-*measured*, before a client's `ga4_data_start`. Query 3 measures how much of the month that affects |
| `sessions_ai` | **excluded** | 30,177 rows carry AI sessions against 78.8M daily rows — too sparse to shape a cluster |
| my ML-07 archetype ladder | **excluded** | circular: clustering on my own grouping rediscovers my own grouping |
| `fact_content_query_90d.*` | **excluded** | fixed 90-day window does not align with my one-month observation window |

**The rule underneath all of it.** A supervised lane asks *"was this knowable before the decision?"* An
unsupervised descriptive lane asks the sibling question: **"was this measured inside the period I claim to
be describing?"** Same discipline, different failure mode — and the failure mode here is quieter, because
a mixed-period cluster still looks perfectly reasonable in a profile table.

## 3. Verify it with queries (grain, counts, missing values, windows)

Three queries, one per contract claim. A contract line without a query next to it is a guess.

### Query 1 of 3 — the grain really is what I said

In [4]:
grain = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {FACT}
    GROUP BY 1, 2, 3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Rows where (report_date, client_hash_id, content_hash_id) repeats: {len(grain)}")
print("Zero rows back => the grain holds: one row = one content item, one client, one day.")
print("That matters here because my unit is the PAGE - if the source grain were finer or")
print("duplicated, my per-page aggregate would silently double-count.\n")
grain

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows where (report_date, client_hash_id, content_hash_id) repeats: 0
Zero rows back => the grain holds: one row = one content item, one client, one day.
That matters here because my unit is the PAGE - if the source grain were finer or
duplicated, my per-page aggregate would silently double-count.



,report_date,client_hash_id,content_hash_id,n


### Query 2 of 3 — my slice's row count and date span

In [5]:
span = con.sql(f"""
    SELECT COUNT(*)                        AS rows_in_month,
           COUNT(DISTINCT content_hash_id) AS content_items,
           COUNT(DISTINCT client_hash_id)  AS clients,
           MIN(report_date)                AS first_day,
           MAX(report_date)                AS last_day,
           COUNT(DISTINCT report_date)     AS distinct_days
    FROM {FACT}
""").df()

print(f"Observation window: month={MONTH}")
print(span.to_string(index=False))
print()
print("Contract claim being checked: the observation window is exactly March 2026.")
print("If first_day/last_day fall outside 2026-03, my partition path is wrong and every")
print("archetype profile below describes a period I did not intend.")

Observation window: month=2026-03
 rows_in_month  content_items  clients  first_day   last_day  distinct_days
       9841378         331437       55 2026-03-01 2026-03-31             31

Contract claim being checked: the observation window is exactly March 2026.
If first_day/last_day fall outside 2026-03, my partition path is wrong and every
archetype profile below describes a period I did not intend.


### Query 3 of 3 — availability, checked with `IS TRUE`

`ga4_data_available = FALSE` marks rows where the GA4 columns were **zero-filled because tracking had not
started**, not measured as zero. For a clustering lane this is the single most dangerous column in the
warehouse: filler zeros do not just bias an average, they **create a cluster**. A group of pages whose only
shared property is "GA4 was not switched on yet" will separate cleanly, profile convincingly, and mean
nothing.

`IS TRUE` rather than `= TRUE` on purpose: `IS TRUE` returns false for NULL instead of returning NULL, so
rows where the flag itself is missing land on the safe side of the filter rather than vanishing.

In [6]:
avail = con.sql(f"""
    SELECT COUNT(*)                                               AS all_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)     AS ga4_available_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_not_available_rows,
           COUNT(*) FILTER (WHERE gsc_impressions > 0)            AS rows_with_impressions,
           COUNT(DISTINCT content_hash_id) FILTER (WHERE ga4_data_available IS TRUE)
                                                                  AS content_with_ga4
    FROM {FACT}
""").df()

r = avail.iloc[0]
print(f"month={MONTH}")
print(f"  all rows                          : {r.all_rows:,}")
print(f"  ga4_data_available IS TRUE        : {r.ga4_available_rows:,} "
      f"({r.ga4_available_rows/r.all_rows*100:.1f}% survive)")
print(f"  NOT TRUE (zero-filled or missing) : {r.ga4_not_available_rows:,} "
      f"({r.ga4_not_available_rows/r.all_rows*100:.1f}%)")
print(f"  rows with any search impression   : {r.rows_with_impressions:,} "
      f"({r.rows_with_impressions/r.all_rows*100:.1f}%)")
print(f"  content items with any GA4 data   : {r.content_with_ga4:,}")
print()
print("Whatever share fails the flag is the share of the panel where an engagement feature")
print("would be fabricated from filler zeros - and in an unsupervised lane, fabricated zeros")
print("do not blur a boundary, they DRAW one. This is why my five features are GSC-only.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

month=2026-03
  all rows                          : 9,841,378
  ga4_data_available IS TRUE        : 413,966 (4.2% survive)
  NOT TRUE (zero-filled or missing) : 9,427,412 (95.8%)
  rows with any search impression   : 3,611,061 (36.7%)
  content items with any GA4 data   : 90,489

Whatever share fails the flag is the share of the panel where an engagement feature
would be fabricated from filler zeros - and in an unsupervised lane, fabricated zeros
do not blur a boundary, they DRAW one. This is why my five features are GSC-only.


## Five features, max — and one line each on *why it is knowable at the decision moment*

For a descriptive lane the "decision moment" is **the last second of 2026-03-31**, the instant the
observation window closes. Every feature below is computed only from rows whose `report_date` falls inside
March, so every one of them describes the period the archetype claims to describe — and nothing else.

| # | Feature | Knowable at the decision moment because… |
|---|---|---|
| 1 | `imp_mar` — March impressions | a sum over March days only; on 31 March it is complete and final. This is the exposure the archetype is about. |
| 2 | `ctr_mar` — March clicks ÷ March impressions | both inputs are March-only sums, so the rate describes March and not a trailing average that reaches into February. |
| 3 | `pos_mar` — impression-weighted average position across March | **this is the starter-CSV fix.** In ML-02 I used `avg_position`, a 90-day mean; a page that moved from rank 30 to rank 5 during the window showed up as a mediocre 17 and clustered with pages that never moved. Weighted over March days only, it describes where the page actually was. |
| 4 | `active_days_mar` — days in March with ≥ 1 impression | counted from March rows only. Separates steady exposure from a one-week spike — the distinction that produced the `Intermittent Exposure` archetype on the starter slice. |
| 5 | `log_clicks_mar` — log1p of March clicks | a March-only sum; logged because click counts are heavy-tailed and one 40,000-click page would otherwise define an axis by itself. |

Five, and no more, on purpose: this card is about proving the contract holds, not about producing the final
archetype set. Note what is **not** here — no engagement, no AI sessions, no query mix, no all-time content
age. Each was excluded by a query above or a window argument, not by preference.

**The heavy cell.** It scans one month partition and returns one small row per page.

In [7]:
MIN_IMPRESSIONS = 300   # policy, carried from ML-02 (the documented "moderate" tier boundary)

feat = con.sql(f"""
    SELECT client_hash_id,
           content_hash_id,
           SUM(gsc_impressions)                                     AS imp_mar,
           SUM(gsc_clicks)                                          AS clk_mar,
           COUNT(*) FILTER (WHERE gsc_impressions > 0)              AS active_days_mar,
           SUM(gsc_avg_position * gsc_impressions)
               FILTER (WHERE gsc_impressions > 0 AND gsc_avg_position > 0)
             / NULLIF(SUM(gsc_impressions)
               FILTER (WHERE gsc_impressions > 0 AND gsc_avg_position > 0), 0) AS pos_mar
    FROM {FACT}
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= {MIN_IMPRESSIONS}
""").df()

print(f"pages with >= {MIN_IMPRESSIONS} impressions in {MONTH}: {len(feat):,}")
print(f"clients represented                                : {feat['client_hash_id'].nunique()}")
assert feat["content_hash_id"].is_unique, "grain broken: the frame must be one row per page"
print("Frame grain check passed: one row = one content item.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

pages with >= 300 impressions in 2026-03: 74,666
clients represented                                : 40
Frame grain check passed: one row = one content item.


In [8]:
d = feat.copy()
d["ctr_mar"] = d["clk_mar"] / d["imp_mar"] * 100
d["log_imp_mar"] = np.log1p(d["imp_mar"])
d["log_clicks_mar"] = np.log1p(d["clk_mar"])
d["consistency_mar"] = d["active_days_mar"] / 31

FEATURES = ["log_imp_mar", "ctr_mar", "pos_mar", "active_days_mar", "log_clicks_mar"]
assert len(FEATURES) == 5, "the card allows five features, max"
print(f"THE FIVE FEATURES: {FEATURES}\n")
print(d[["content_hash_id"] + FEATURES].describe().round(2).to_string())
print(f"\nnulls per feature:\n{d[FEATURES].isna().sum().to_string()}")
print("\n(pos_mar can be null where a page had impressions but no recorded position -")
print(" that is 'no data', not rank zero, and it is dropped rather than imputed to 0.)")
d = d.dropna(subset=FEATURES).reset_index(drop=True)
print(f"\nframe after dropping null-position pages: {len(d):,} pages")

THE FIVE FEATURES: ['log_imp_mar', 'ctr_mar', 'pos_mar', 'active_days_mar', 'log_clicks_mar']

       log_imp_mar   ctr_mar   pos_mar  active_days_mar  log_clicks_mar
count     74666.00  74666.00  74666.00         74666.00        74666.00
mean          7.41      0.27     12.20            29.28            1.47
std           1.16      0.37     12.14             4.23            1.26
min           5.71      0.00      0.02             1.00            0.00
25%           6.45      0.02      4.35            30.00            0.69
50%           7.25      0.17      7.25            31.00            1.39
75%           8.22      0.37     15.94            31.00            2.30
max          13.33      9.83     91.51            31.00            8.64

nulls per feature:
log_imp_mar        0
ctr_mar            0
pos_mar            0
active_days_mar    0
log_clicks_mar     0

(pos_mar can be null where a page had impressions but no recorded position -
 that is 'no data', not rank zero, and it is dropped r

## The trap — one grouping-derived column, on purpose

The lesson from notebook 02, translated into an unsupervised lane. There is no label here, so the
supervised version of the trap — feeding the model a future outcome — has no direct equivalent. The
unsupervised equivalent is **feeding the model the grouping you are trying to discover**, and it is more
seductive because the resulting score looks like validation rather than like cheating.

I add **exactly one** column: my own hand-written archetype ladder from ML-07, encoded as a number. Then I
watch two things move — the silhouette, and the agreement with the ladder itself.

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

SEED, K = 42, 6

# The ML-07 ladder, rebuilt from March-only columns (no days_since_last_update in this frame,
# so this is a simplified three-rung version - the point is that it is MY grouping, not the world's).
def ladder(r):
    if r["imp_mar"] >= 3000 and r["pos_mar"] <= 10: return 0    # champion-ish
    if r["active_days_mar"] < 15:                   return 1    # intermittent
    if r["ctr_mar"] >= 0.30:                        return 2    # converting
    return 3                                                     # catch-all
d["my_ladder"] = d.apply(ladder, axis=1)
print("my hand-written grouping:", d["my_ladder"].value_counts().sort_index().to_dict(), "\n")

samp = np.random.RandomState(0).choice(len(d), min(5000, len(d)), replace=False)

def quick_score(cols, label):
    Xs = StandardScaler().fit_transform(d[cols])
    lab = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit_predict(Xs)
    sil = silhouette_score(Xs[samp], lab[samp])
    ari = adjusted_rand_score(d["my_ladder"], lab)
    print(f"  {label:36s} silhouette = {sil:.3f}   ARI vs my ladder = {ari:.3f}")
    return sil, ari, lab

print("HONEST - the five features from the contract:")
honest_sil, honest_ari, honest_lab = quick_score(FEATURES, "5 contract features")

my hand-written grouping: {0: 15289, 1: 1774, 2: 16254, 3: 41349} 

HONEST - the five features from the contract:
  5 contract features                  silhouette = 0.276   ARI vs my ladder = 0.229


In [10]:
LEAK = "my_ladder"
print("LEAKED - the same five features, plus my own grouping as a sixth:")
leak_sil, leak_ari, leak_lab = quick_score(FEATURES + [LEAK], f"5 features + {LEAK}")

print()
print(f"silhouette      {honest_sil:.3f}  ->  {leak_sil:.3f}   ({leak_sil-honest_sil:+.3f})")
print(f"ARI vs ladder   {honest_ari:.3f}  ->  {leak_ari:.3f}   ({leak_ari-honest_ari:+.3f})")
print()
print("Look at the SECOND number, not the first. ARI vs my ladder climbing toward 1.0 means")
print("the clustering is reproducing the grouping I handed it. If I reported that as")
print("'the clusters validate my archetype ladder', it would be a circular result: the")
print("agreement was in the input.")
print()
if leak_sil - honest_sil < 0.02:
    print("NOTE: the silhouette barely moved this time. That does NOT make the column harmless -")
    print("the ARI is the tell. A leak that does not move your headline metric is more dangerous,")
    print("not less, because nothing warns you. Judge it by whether the column encodes the answer,")
    print("never by whether the score jumped.")

LEAKED - the same five features, plus my own grouping as a sixth:
  5 features + my_ladder               silhouette = 0.325   ARI vs my ladder = 0.453

silhouette      0.276  ->  0.325   (+0.049)
ARI vs ladder   0.229  ->  0.453   (+0.224)

Look at the SECOND number, not the first. ARI vs my ladder climbing toward 1.0 means
the clustering is reproducing the grouping I handed it. If I reported that as
'the clusters validate my archetype ladder', it would be a circular result: the
agreement was in the input.



In [11]:
# Delete it. Not comment it out - delete it, so it cannot come back by accident.
d = d.drop(columns=[LEAK])
assert LEAK not in d.columns, "the grouping column is still here"
assert LEAK not in FEATURES, "the grouping column got into the feature list"
assert "client_hash_id" not in FEATURES, "client_hash_id must stay a check, never a feature"
print(f"Dropped '{LEAK}'. Guards passed: no pre-computed grouping remains in the feature set.\n")

print("THE NUMBERS I KEEP - five contract features, nothing else:")
print(f"  silhouette    = {honest_sil:.3f}")
print(f"  ARI vs ladder = {honest_ari:.3f}   <- moderate agreement, INDEPENDENTLY arrived at.")
print()
print("That second number is only meaningful because the ladder was never an input. An")
print("honest 0.3 that I can interpret beats a manufactured 0.9 that I cannot.")

Dropped 'my_ladder'. Guards passed: no pre-computed grouping remains in the feature set.

THE NUMBERS I KEEP - five contract features, nothing else:
  silhouette    = 0.276
  ARI vs ladder = 0.229   <- moderate agreement, INDEPENDENTLY arrived at.

That second number is only meaningful because the ladder was never an input. An
honest 0.3 that I can interpret beats a manufactured 0.9 that I cannot.


## 4. Data limits — what this slice can never tell me

**The named limitation of my slice: one calendar month of an unbalanced panel, which means an "archetype"
here is a snapshot, not a trajectory — and for a lane about *what kind of page this is*, that is the
limitation that matters most.**

A page that has been quietly stable for eighteen months and a page that collapsed into that state three
weeks ago look identical in a March aggregate. They are not the same asset and they do not deserve the same
treatment. The query below measures how much of my panel is even *capable* of supporting a trajectory
answer.

In [12]:
cl = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM {DIM_CLIENTS}
""").df()
cl["gsc_data_start"] = pd.to_datetime(cl["gsc_data_start"])

mine = cl[cl["client_hash_id"].isin(d["client_hash_id"].unique())].copy()
cutoff = pd.Timestamp(f"{MONTH}-01")
mine["months_before_window"] = ((cutoff - mine["gsc_data_start"]).dt.days / 30.44).round(1)

print(f"Clients in my frame: {len(mine)}")
print(f"  history before the observation window (months): "
      f"min {mine['months_before_window'].min():.1f}, "
      f"median {mine['months_before_window'].median():.1f}, "
      f"max {mine['months_before_window'].max():.1f}")
print(f"  clients with < 3 months of history before 2026-03-01: "
      f"{int((mine['months_before_window'] < 3).sum())}")
print(f"  clients with >= 12 months (enough to ask 'has this page ALWAYS been like this?'): "
      f"{int((mine['months_before_window'] >= 12).sum())}")
print()
print("Only the clients in that last count can support a trajectory question at all. For the")
print("rest, 'this page has plateaued' is unfalsifiable - there is no earlier period to")
print("compare against, and a plateau and a fresh start look the same from inside one month.")

Clients in my frame: 40
  history before the observation window (months): min -0.9, median 4.2, max 13.1
  clients with < 3 months of history before 2026-03-01: 17
  clients with >= 12 months (enough to ask 'has this page ALWAYS been like this?'): 3

Only the clients in that last count can support a trajectory question at all. For the
rest, 'this page has plateaued' is unfalsifiable - there is no earlier period to
compare against, and a plateau and a fresh start look the same from inside one month.


### The rest of the honest list

- **Clusters are a lens, not natural kinds.** Carried forward from ML-02 and not softened: k is a choice,
  the silhouette curve is nearly flat across a range of k, and a different method finds a different
  partition of the same data.
- **No engagement dimension.** Query 3 measured why. Until I can filter on `ga4_data_available IS TRUE`
  and still have a panel worth clustering, the engagement half of the archetype picture is missing —
  which means "engaged" and "not engaged" are not distinctions this contract can draw.
- **No content metadata in the features.** Length, keyword economics and intent are held out for the
  external-coherence check ML-03 committed to. That is a deliberate trade: I lose descriptive richness in
  the clusters and gain an independent way to test whether they are real.
- **No consolidation view.** If two pages split the same demand, this contract sees two ordinary pages.
  That needs `url_hash_id` / `keyword_hash_id` grouping, which is a later card.
- **Nothing here is causal or predictive.** A cluster describes how a page behaved in March. It does not
  say what the page will do, and an action attached to a cluster is a hypothesis about where a human should
  look — never a claim that the action will work.
- **`prune` remains unavailable.** ML-02 committed to never recommending removal on cluster membership
  alone. One month of data makes that commitment stronger, not weaker.

### What the contract hands to a human, in one sentence
> A small set of behavioural archetypes describing how each page performed during one fully-measured month,
> each with a suggested treatment and an explicit note on what would make that treatment wrong — for a
> content lead setting policy, not for a system acting automatically.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — pseudonymized hash ids only, and the HF token is
      read from Colab Secrets, never typed into a cell
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Card checklist
- [x] **Five plain-words contract answers** — section 1
- [x] **Exactly three verification queries with outputs visible** — Query 1 (grain), Query 2 (row count +
      date span), Query 3 (availability via `IS TRUE`)
- [x] **Five features, each with an "available when?" line** — the table above the heavy cell
- [x] **The deliberate leak shown and removed** — my own archetype ladder added as a sixth column, scored,
      then dropped with an assert. The tell was **ARI**, not silhouette.
- [x] **One named limitation** — one month is a snapshot, not a trajectory, measured against
      `dim_clients.gsc_data_start`

### What I owe ML-06 / ML-07
1. **Two more month partitions** — cluster February, March and April separately and measure how many pages
   keep the same archetype. An archetype that a page holds for one month and not the next is a mood, not a
   type.
2. **Bring engagement back if the flag allows it** — re-run Query 3 on a month with better GA4 coverage and
   see whether a usable panel survives.
3. **Add the held-out variables as validators, never as features** — length, keyword economics and intent
   are how I test the clusters from outside.
4. **Test the position fix** — `pos_mar` is computed in-window here for the first time. Compare the
   resulting archetypes against the starter-CSV versions and see how much the 90-day mean was distorting.